In [1]:
import os

In [2]:
os.chdir("../")

In [3]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

e:\medical_chat_bot\nayak\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# extract text from pdf files

In [4]:
def load_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    
    documents = loader.load()
    return documents

In [5]:
extracted_data=load_files(r"E:\medical_chat_bot\data")
print(len(extracted_data))

637


In [6]:
from typing import List
from langchain_core.documents import Document 


In [7]:
def filter_to_minimal_docs(docs:list[Document])->List[Document]:
    minimal_docs:List[Document]=[]
    for doc in docs:
        src=doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source":src}
            )
        )
    return minimal_docs

minimal_docs=filter_to_minimal_docs(extracted_data)
        

# split the documnet into smaller chunks

In [8]:
def text_split(minimal_docs):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    text_chunk=text_splitter.split_documents(minimal_docs)
    return text_chunk

text_chunks=text_split(minimal_docs)
print(f"Number of chunks: {len(text_chunks)}")

Number of chunks: 5860


# converting text into embeddings

In [9]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [10]:

def download_embeddings():
    model_name = "BAAI/bge-small-en-v1.5"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings

embedding = download_embeddings()


    

C:\Users\User\AppData\Local\Temp\ipykernel_22128\2574293956.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1197.06it/s]


In [11]:
vector = embedding.embed_query("shoban nayak jatoth")
print(len(vector))

384


In [32]:
from dotenv import load_dotenv
load_dotenv()

PINECONE_API_KEY=os.getenv("API_KEY")
GOOGLE_API_KEY = os.getenv("GEN_API")

os.environ["PINECONE_API_KEY"]=PINECONE_API_KEY
os.environ["GEM_API_KEY"]=GOOGLE_API_KEY

In [15]:
from pinecone import Pinecone

pinecone_api_key = PINECONE_API_KEY 

pc = Pinecone(api_key=pinecone_api_key)

# create database


In [23]:
from pinecone import ServerlessSpec
index_name="medical-chat-bot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws",region="us-east-1")
        
    )

index=pc.Index(index_name)

In [24]:
from langchain_pinecone import PineconeVectorStore
doc_search=PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embedding,
    index_name=index_name
)

# Load existing index

In [25]:
from langchain_pinecone import PineconeVectorStore
doc_search=PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

In [27]:
retriver=doc_search.as_retriever(
    search_type="similarity",
    search_kawargs={"k":3}
)

In [28]:
retrived_docs=retriver.invoke("what is acne")
retrived_docs

[Document(id='d435687f-e529-45bb-9313-efec31c4ddc0', metadata={'source': 'E:\\medical_chat_bot\\data\\Medical_book.pdf'}, page_content='Acidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when the\npores of the skin become clogged with oil, dead skin\ncells, and bacteria.\nDescription\nAcne vulgaris, the medical term for common acne, is\nthe most common skin disease. It affects nearly 17 million\npeople in the United States. While acne can arise at any'),
 Document(id='eb6b6b1d-5e71-463c-9b4c-31b57de7df7c', metadata={'source': 'E:\\medical_chat_bot\\data\\Medical_book.pdf'}, page_content='of the brain. Make sure the physician knows if tetracy-\ncline is being used to treat acne or another infection.\nNancy Ross-Flanigan\nKEY TERMS\nAcne—A skin condition in which raised bumps,\npimples, and cysts form on the face, neck, shoul-\nders and upper back

# Model

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
model=ChatGoogleGenerativeAI(model="gemini-2.5-flash")


ImportError: cannot import name 'ContextOverflowError' from 'langchain_core.exceptions' (e:\medical_chat_bot\nayak\Lib\site-packages\langchain_core\exceptions.py)

In [2]:
from langchain.chains.retrieval import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [3]:
system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [4]:
question_answer_chain = create_stuff_documents_chain(model, prompt)
rag_chain = create_retrieval_chain(retriver, question_answer_chain)

NameError: name 'model' is not defined

In [ ]:
response = rag_chain.invoke({"input": "what is Acromegaly and gigantism?"})
print(response["answer"])